# 13 · A3 stage 4: demo data

For each demo area (the four pools' test tiles), every 3rd pixel (a 90 m grid): the stage 2 model's projected
vector (notebook 11's best text encoder), the per-class AlphaEarth classifier probabilities (notebook 12), the
official land class (for the facts panel), the POWER cell indices, and a lon/lat display grid for the map. The
notebook can run as soon as stage 2 (notebook 11) exists; a method whose inputs do not exist yet is written down
as **not ready** with the reason, and the demo refuses it with that reason (it never substitutes another method).
Run it again after notebook 12 to add the classifier method and the best combination. The "best combination" is
chosen from the stage 3 verdicts by a rule fixed in A3.2:

- the classifier per class if C3 found it better;
- otherwise the aligned space for land with POWER numbers for climate if C2 found numbers better;
- otherwise the aligned space alone.

Start the demo with `uv run python -m askmap.demo.serve` (port in `configs/aether_us.yaml`).

In [1]:
import json
import time

import numpy as np
import pandas as pd

from askmap.catalogue import area_classes
from askmap.config import PROJECT_ROOT, data_root, load_config
from askmap.layers import SOLAR_LATER, official_leaf_map
from askmap.plan import AreaContext, execute
from askmap.power import add_derived, fetch_daily, pixel_cells
from askmap.tiles import area_tiles, pixel_lonlat
from askmap.universe import Universe, box_of

cfg = load_config(PROJECT_ROOT / "configs" / "feasibility.yaml")
Q = load_config(PROJECT_ROOT / "configs" / "queries.yaml")["queries"]
RW = load_config(PROJECT_ROOT / "configs" / "reworded.yaml")["reworded"]
ROOT = data_root()
RES = PROJECT_ROOT / "outputs" / "results"
FIG = PROJECT_ROOT / "outputs" / "figures"
CL, P, SC = cfg["classes"], cfg["power"], cfg["scoring"]
LEAVES, GROUPS = CL["leaves"], CL["groups"]
CATALOGUE = area_classes(cfg)
T_START = time.time()
cells = pd.read_parquet(ROOT / "power" / "universe_cells.parquet")
daily = {
    box: add_derived(fetch_daily(P, b, ROOT / "power" / f"universe_{box}_daily.nc"), P["low_wind_sun_pct"])
    for box, b in P["boxes"].items()
}
uni = Universe(cells=cells, daily=daily)
SEEN = ["north_dakota", "north_carolina"]
HELD = cfg["heldout_pools"]  # seen area -> held-out state
POOL_Q = {
    "north_dakota": "north_dakota",
    "north_carolina": "north_carolina",
    "minnesota": "north_dakota",
    "georgia": "north_carolina",
}  # pool -> area whose questions it answers
T = {}
for area in POOL_Q:
    for name, (tile, role) in area_tiles(cfg["areas"][area], cfg["grid"]).items():
        z = np.load(ROOT / "tiles" / f"{area}_{name}.npz")
        solar = z["solar"] if "solar" in z.files else None
        nwi = z["nwi"] if "nwi" in z.files else None
        valid = np.isfinite(z["ae"]).all(axis=0)
        if solar is not None:
            valid &= solar != SOLAR_LATER
        lon, lat = pixel_lonlat(tile)
        box = box_of(float(lon.mean()), float(lat.mean()), P["boxes"])
        iy, ix = pixel_cells(daily[box].lat.values, daily[box].lon.values, lat, lon)
        T[name] = {
            "area": area,
            "role": role,
            "valid": valid,
            "box": box,
            "iy": iy,
            "ix": ix,
            "cell": uni.index(box, iy, ix),
            "official": official_leaf_map(z["cdl"], LEAVES, CL, nwi=nwi, solar=solar),
        }
        if role == "test":
            T[name]["ae_map"] = np.load(ROOT / "maps" / f"ae_{name}.npz")["leaf"]
            T[name]["planted_map"] = np.load(ROOT / "maps" / f"planted_{name}.npz")["leaf"]
CTX, TILES = {}, {}
for area in POOL_Q:
    names = [n for n, t in T.items() if t["area"] == area]
    ds = daily[T[names[0]]["box"]]
    ac = np.zeros((ds.sizes["lat"], ds.sizes["lon"]), bool)
    for n in names:
        ac[T[n]["iy"], T[n]["ix"]] = True
    CTX[area] = AreaContext(
        leaves=LEAVES,
        groups=GROUPS,
        res_m=cfg["grid"]["res_m"],
        power=ds,
        area_cells=ac,
        pixel_cell={n: (T[n]["iy"], T[n]["ix"]) for n in names},
    )
    TILES[area] = {
        "train": [n for n in names if T[n]["role"] == "train"],
        "test": [n for n in names if T[n]["role"] == "test"],
    }


def answer(plan, pool, maps="official"):
    """Answer of a plan in a pool's test tiles, as one vector over their valid pixels (fixed tile order)."""
    tiles = TILES[pool]["test"]
    key = "official" if maps == "official" else f"{maps}_map"
    got = execute(plan, {n: T[n][key] for n in tiles}, CTX[pool])
    return np.concatenate([got[n][T[n]["valid"]] for n in tiles])


def question_text(q, wording, pool):
    text = q["question"] if wording != "reworded" else RW[q["id"]]
    if pool in ("minnesota", "georgia"):
        text = text.replace("North Dakota", "Minnesota").replace("North Carolina", "Georgia")
    return text


def n_extra(plan):
    return len(plan["near"]) + len(plan["climate"]) + int(plan["land"]["min_patch_ha"] is not None)


print(f"{len(T)} tiles loaded; pools {dict((a, TILES[a]['test']) for a in POOL_Q)}")

18 tiles loaded; pools {'north_dakota': ['Kidder', 'Foster', 'Eddy'], 'north_carolina': ['NC4', 'NC5', 'NC6'], 'minnesota': ['MN1', 'MN2', 'MN3'], 'georgia': ['GA1', 'GA2', 'GA3']}


In [2]:
import os

import torch

from askmap.aether_us import Projector, project
from askmap.demo.core import display_index
from askmap.landmaps import land_probs

AU = load_config(PROJECT_ROOT / "configs" / "aether_us.yaml")
DM, MCFG = AU["demo"], AU["model"]
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
V11 = json.loads((RES / "11_verdicts.json").read_text())
V12 = json.loads((RES / "12_verdicts.json").read_text()) if (RES / "12_verdicts.json").exists() else None
BEST_ENC = V11["best_encoder"]
OUT = ROOT / "aether_us"
DEMO = ROOT / "demo"
DEMO.mkdir(exist_ok=True)
ck = torch.load(OUT / f"projector_{BEST_ENC}.pt", weights_only=False)
MU, SD = ck["mu"], ck["sd"]
dim = np.load(OUT / f"text_{BEST_ENC}.npz")["vectors"].shape[1]
model = Projector(len(MU), dim, MCFG["layers"], MCFG["hidden"]).to(device)
model.load_state_dict(ck["state"])
NOT_READY = {}
# The land maps are built in stages (notebook 12: one-vs-rest tag classifiers; 20: one multinomial model; 23:
# neighbourhood context with more training tiles). This notebook uses the configured method when its files
# exist and otherwise the latest stage that does, saying so; it runs again after the later notebooks.
LAND_METHOD = DM["land_method"]  # one setting for every class (amendment A5)
STAGE_FILES = {
    "one_vs_rest": ["tag_classifiers.npz"],
    "multinomial": ["land_methods.npz"],
    "context_more": ["land_methods.npz", "land_methods_a7.npz"],
}
if LAND_METHOD not in STAGE_FILES:
    raise ValueError(f"unknown land method {LAND_METHOD!r}")
ready = [m for m, files in STAGE_FILES.items() if all((OUT / f).exists() for f in files)]
if not ready:
    raise FileNotFoundError("notebook 12 has not written tag_classifiers.npz")
if LAND_METHOD not in ready:
    print(
        f"land method {LAND_METHOD!r} needs {STAGE_FILES[LAND_METHOD]}, not all built yet: "
        f"using {ready[-1]!r}"
    )
    LAND_METHOD = ready[-1]


def load_stage(name):
    arrays = dict(np.load(OUT / name))
    if list(arrays.pop("leaves")) != LEAVES:
        raise ValueError(f"{name} leaves differ from the configured leaves")
    return arrays


if LAND_METHOD == "one_vs_rest":
    TC = load_stage("tag_classifiers.npz")
    LAND = {"ovr_coef": TC["coef"], "ovr_intercept": TC["intercept"]}
elif LAND_METHOD == "multinomial":
    LAND = load_stage("land_methods.npz")
else:
    LAND = load_stage("land_methods.npz") | load_stage("land_methods_a7.npz")
SUMMARY = np.load(ROOT / "climate" / "enc_summary.npy")
if V12 is None:
    BEST = None
elif V12["C3"]["winner"] == "classifier per tag":
    BEST = "classifier"
elif V12["C2"]["numbers_win"]:
    BEST = "aligned_land_power_climate"
else:
    BEST = "aligned"
S = DM["stride"]
print(f"encoder {BEST_ENC}; best combination: {BEST}; land maps: {LAND_METHOD}; not ready: {NOT_READY}")

encoder clip; best combination: classifier; land maps: context_more; not ready: {}


In [3]:
meta = {
    "leaves": LEAVES,  # the class list the data is built with; the demo serves this list, not the config's
    "stride": S,
    "top_share": 0.02,
    "encoder": AU["text_encoders"][BEST_ENC],
    "encoder_kind": {"clip": "clip", "modern": "sentence_transformer"}[BEST_ENC],
    "best": BEST,
    "land_method": LAND_METHOD,
    "examples": DM["examples"],
    "not_ready": NOT_READY,
    "areas": {},
}
for area in DM["areas"]:
    tiles = area_tiles(cfg["areas"][area], cfg["grid"])
    entry = {"label": cfg["question_areas"][area].split(" (")[0], "box": None, "tiles": []}
    for n in TILES[area]["test"]:
        tile = tiles[n][0]
        ae_full = np.load(ROOT / "tiles" / f"{area}_{n}.npz")["ae"].astype(np.float32)
        ae = ae_full[:, ::S, ::S]
        valid = T[n]["valid"][::S, ::S]
        cell = T[n]["cell"][::S, ::S]
        x = ((np.c_[ae.reshape(64, -1).T[valid.ravel()], SUMMARY[cell[valid]]] - MU) / SD).astype(np.float32)
        z = project(model, x, device).astype(np.float16)
        if LAND is None:
            probs = np.zeros((len(LEAVES), *valid.shape), np.float16)
        else:
            probs = land_probs(LAND_METHOD, LAND, ae_full, S).astype(np.float16)
        index, bounds = display_index(tile, S, 320)
        np.savez_compressed(
            DEMO / f"{n}.npz",
            index=index,
            valid=valid,
            z=z,
            probs=probs,
            leaf=T[n]["official"][::S, ::S].astype(np.uint8),
            iy=T[n]["iy"][::S, ::S],
            ix=T[n]["ix"][::S, ::S],
        )
        entry["box"] = T[n]["box"]
        entry["tiles"].append({"name": n, "bounds": [float(b) for b in bounds]})
    meta["areas"][area] = entry
(DEMO / "meta.json").write_text(json.dumps(meta, indent=1))
sizes = {p.name: p.stat().st_size for p in DEMO.glob("*.npz")}
print(f"{len(sizes)} tiles, {sum(sizes.values()) / 1e6:.0f} MB")

12 tiles, 846 MB


## Timing check of the three example questions (engine without the web server)

In [4]:
from askmap.aether_us import TextEncoder
from askmap.catalogue import catalogue_text
from askmap.demo.engine import Engine
from askmap.demo.serve import load_tiles, make_contexts, make_planner
from askmap.llm import make_client, system_prompt, translate

CAP = load_config(PROJECT_ROOT / "configs" / "captions.yaml")
enc = TextEncoder(meta["encoder"], meta["encoder_kind"], device, os.environ["HF_HOME"])
llm = cfg["llm"]
client = make_client(llm["base_url"])
system = system_prompt((PROJECT_ROOT / "configs" / "language.md").read_text(), catalogue_text(cfg))
planner = make_planner(
    cfg,
    lambda q, path: translate(client, llm["models"]["main"], system, q, path, llm),
    PROJECT_ROOT / "outputs" / "llm" / "demo",
)
engine = Engine(
    load_tiles(DEMO, meta),
    LEAVES,
    enc,
    planner,
    make_contexts(cfg, meta, DEMO),
    BEST,
    CAP,
    top_share=meta["top_share"],
    not_ready=NOT_READY,
)
print("not ready:", engine.not_ready)
rows = []
for q in DM["examples"]:
    # "best" maps to another method, which may itself be not ready on a first pass
    ready = [
        m
        for m in ("best", "aligned", "classifier")
        if (engine.best if m == "best" else m) not in engine.not_ready and m not in engine.not_ready
    ]
    for method in ready:
        r = engine.search(q, "north_carolina", method)
        rows.append(
            {
                "question": q,
                "method": method,
                "used": r["used"],
                "seconds": r["seconds"],
                "shown": r["facts"]["shown_pixels"],
                "top_land": r["facts"]["land_of_shown_pixels"][:2],
            }
        )
pd.DataFrame(rows)

/Volumes/disco2/github/climate-embeddings-crop-integrity/ask-the-map/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1029 22:52:21.687000 16612 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 23576.53it/s]

not ready: {}


,question,method,used,seconds,shown,top_land
0,wetlands next to crop fields,best,classifier,0.10,9740,"[{'class': 'wetland', 'share': 0.637}, {'class..."
1,wetlands next to crop fields,aligned,aligned,0.34,224,"[{'class': 'wetland', 'share': 0.982}, {'class..."
2,wetlands next to crop fields,classifier,classifier,0.04,9740,"[{'class': 'wetland', 'share': 0.637}, {'class..."
3,corn fields where July 2021 had at least 5 day...,best,classifier,0.01,0,[]
4,corn fields where July 2021 had at least 5 day...,aligned,aligned,0.15,2984,"[{'class': 'corn', 'share': 0.917}, {'class': ..."
5,corn fields where July 2021 had at least 5 day...,classifier,classifier,0.04,0,[]
6,grassland near an existing solar farm,best,classifier,0.01,278,"[{'class': 'grassland', 'share': 0.363}, {'cla..."
7,grassland near an existing solar farm,aligned,aligned,0.08,69,"[{'class': 'grassland', 'share': 0.594}, {'cla..."
8,grassland near an existing solar farm,classifier,classifier,0.06,278,"[{'class': 'grassland', 'share': 0.363}, {'cla..."


In [5]:
print(f"Run time {(time.time() - T_START) / 60:.1f} min")

Run time 2.9 min
